# 03: Infrastructure Damage Assessment

**Methodology:** Based on the 2023 SPIE publication: *"Harvesting remote sensing observations for quantifying burned area and built-up losses from the 2021 wildfires in Greece."*

This final Phase 1 notebook assesses the physical impact of the wildfires on human infrastructure. By performing spatial intersections between the vectorized burn scars and foundational infrastructure datasets (OpenStreetMap roads and Microsoft GlobalML building footprints), we calculate the exact length of affected road networks and the total surface area of exposed buildings.

In [ ]:
import os
import geopandas as gpd
import matplotlib.pyplot as plt

print("Geospatial libraries loaded successfully.")

## 2. Configuration & File Paths
We load the burn scar polygons generated in the previous notebook, alongside our infrastructure datasets. 

*Note: Ensure you have exported your road network and building footprint data for your Area of Interest (AOI) into the `data/` folder.*

In [ ]:
# --- File Paths ---
DATA_DIR = "data"

# Input Burn Scars (From Notebook 02)
BURNED_POLYGONS = os.path.join(DATA_DIR, "burned_area_polygons.geojson")

# Infrastructure Datasets (Replace with your actual OSM/Microsoft GlobalML extracts)
ROADS_FILE = os.path.join(DATA_DIR, "aoi_roads.geojson")
BUILDINGS_FILE = os.path.join(DATA_DIR, "aoi_buildings.geojson")

print("File paths configured.")

## 3. Data Loading & Coordinate Reference System (CRS) Alignment
To perform accurate spatial intersections and calculate real-world distances (meters/kilometers), all datasets must be projected into a metric Coordinate Reference System. 

For Greece, the standard projected CRS is **EPSG:2100 (Greek Grid)**. We will load the data and immediately reproject it to ensure our geometric math is perfectly accurate.

In [ ]:
# Define target metric CRS (Greek Grid)
TARGET_CRS = "EPSG:2100"

# Load and reproject Burn Scars
burn_gdf = gpd.read_file(BURNED_POLYGONS).to_crs(TARGET_CRS)
print(f"Burn Scars loaded. CRS: {burn_gdf.crs}")

# Load and reproject Roads
roads_gdf = gpd.read_file(ROADS_FILE).to_crs(TARGET_CRS)
print(f"Roads loaded. Total features: {len(roads_gdf)}")

# Load and reproject Buildings
buildings_gdf = gpd.read_file(BUILDINGS_FILE).to_crs(TARGET_CRS)
print(f"Buildings loaded. Total features: {len(buildings_gdf)}")

## 4. Road Network Exposure
We use `geopandas.overlay(how='intersection')` to clip the road network exactly to the boundaries of the burn scars. Then, we calculate the length of the resulting clipped line geometries.

In [ ]:
print("Calculating road network exposure...")

# Perform spatial intersection
damaged_roads = gpd.overlay(roads_gdf, burn_gdf, how='intersection')

# Calculate total length in meters, then convert to kilometers
damaged_roads['length_m'] = damaged_roads.geometry.length
total_road_length_km = damaged_roads['length_m'].sum() / 1000

print(f"✅ Total road network exposed to fire: {total_road_length_km:.2f} km")

## 5. Building Footprint Exposure
Similarly, we intersect the building footprints with the burn scars. Because buildings are polygons, we calculate the exposed surface area in square kilometers.

In [ ]:
print("Calculating building footprint exposure...")

# Perform spatial intersection
damaged_buildings = gpd.overlay(buildings_gdf, burn_gdf, how='intersection')

# Calculate total area in square meters, then convert to square kilometers
damaged_buildings['area_sqm'] = damaged_buildings.geometry.area
total_building_area_sqkm = damaged_buildings['area_sqm'].sum() / 1_000_000

print(f"✅ Total building area exposed to fire: {total_building_area_sqkm:.4f} sq km")
print(f"✅ Total individual buildings affected: {len(damaged_buildings)}")

## 6. Visualization of Damaged Infrastructure
A comprehensive map displaying the burn scar boundaries alongside the highly impacted roads and buildings.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 12))

# Plot base burn area (light red/pink)
burn_gdf.plot(ax=ax, facecolor='#ff9999', edgecolor='darkred', alpha=0.5, label='Burn Scar')

# Plot damaged roads (black lines)
if not damaged_roads.empty:
    damaged_roads.plot(ax=ax, color='black', linewidth=1.5, label='Affected Roads')

# Plot damaged buildings (bright yellow)
if not damaged_buildings.empty:
    damaged_buildings.plot(ax=ax, color='yellow', edgecolor='orange', linewidth=0.5, label='Affected Buildings')

ax.set_title(f"Infrastructure Damage Assessment\nExposed Roads: {total_road_length_km:.2f} km | Exposed Buildings: {len(damaged_buildings)}", fontsize=14)
ax.set_axis_off()

# Add a simple legend manually
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

burn_patch = mpatches.Patch(color='#ff9999', label='Burn Scar')
road_line = mlines.Line2D([], [], color='black', label='Exposed Roads')
bldg_patch = mpatches.Patch(color='yellow', label='Exposed Buildings')
ax.legend(handles=[burn_patch, road_line, bldg_patch], loc='upper right')

plt.tight_layout()
plt.show()